# BEATs: frozen vs fine-tuned encoder — five-fold cross-validation

Compare two **identical BEATs + patch-reconstruction decoder architectures** on Intel welding anomaly detection. Run five matched folds, each starting both variants from the same pretrained encoder and decoder weights. Cross-validation uses the combined published TRAIN + VAL development pool; the published TEST split remains held out until all folds finish training. The baseline trains only the decoder; the other trains the encoder and decoder together.

BEATs in this repository is an encoder, so this notebook adds a shared decoder design to make an autoencoder. It reconstructs BEATs-normalized log-mel patches, **not audio waveforms**. Higher reconstruction MSE means more anomalous audio. Fine-tuning is an experiment; improvement is not assumed.

Use the project's Python environment with `requirements.txt`, plus `jupyter`, `nbformat`, `ipykernel`, and `matplotlib`. Select that environment as the notebook kernel. No downloads or package installation run automatically.

Set the checkpoint/data paths below, then **Run All**. For an executable plumbing check without pretrained weights or dataset files, set `SMOKE_TEST = True`; those synthetic results have no scientific meaning.

In [13]:
from pathlib import Path
import copy
import gc
import json
import random
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.compliance.kaldi as ta_kaldi
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/encoders.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Launch this notebook from the AI360-Project repository.")
sys.path.insert(0, str(ROOT / "src"))
from encoders import load_beats

SMOKE_TEST = False
SEED = 42
CHECKPOINT = ROOT / "models/BEATs_iter3.pt"
DATA_ROOT = ROOT / "data/intel_robotic_welding_extracted/raid/intel_robotic_welding_dataset"
MANIFEST = DATA_ROOT / "manifest.csv"
OUTPUT_DIR = ROOT / "artifacts/encoder_test_cv5"
SAMPLE_RATE = 16_000
CLIP_SECONDS = 2.0
HOP_SECONDS = 2.0
BATCH_SIZE = 4
EPOCHS = 5
N_FOLDS = 5
TUNING_FRACTION = 0.2
ENCODER_LR = 1e-5
DECODER_LR = 1e-3
WEIGHT_DECAY = 1e-4
BOTTLENECK = 128
TARGET_FPR = 0.05
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if SMOKE_TEST:
    CLIP_SECONDS, HOP_SECONDS, BATCH_SIZE, EPOCHS, BOTTLENECK = 0.5, 0.5, 2, 2, 8
    OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="encoder-cv5-smoke-"))
    torch.set_num_threads(2)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f"Device: {DEVICE}; mode: {'synthetic smoke test' if SMOKE_TEST else 'real experiment'}")


Device: cuda; mode: real experiment


## Load the common initialization

The real experiment requires the local checkpoint documented in `HUMANS.md` (default: `models/BEATs_iter3.pt`). The existing loader reads its `cfg` and `model` keys. A classification head, if present, is discarded: both variants use only the BEATs encoder and the same new decoder. Smoke mode uses a tiny randomly initialized BEATs with the repository's actual implementation.

In [14]:
if SMOKE_TEST:
    sys.path.insert(0, str(ROOT / "src/beats"))
    from BEATs import BEATs, BEATsConfig
    base_encoder = BEATs(BEATsConfig(dict(
        input_patch_size=16, embed_dim=16, encoder_embed_dim=16,
        encoder_ffn_embed_dim=32, encoder_attention_heads=2,
        encoder_layers=1, conv_pos=16, conv_pos_groups=2,
        dropout=0.0, attention_dropout=0.0,
    ))).eval()
else:
    if not CHECKPOINT.is_file():
        raise FileNotFoundError(
            f"Missing BEATs checkpoint: {CHECKPOINT}. Set CHECKPOINT to your local "
            "BEATs checkpoint (see HUMANS.md), or set SMOKE_TEST=True for a synthetic check."
        )
    base_encoder = load_beats(CHECKPOINT)

base_encoder.predictor = None
base_encoder.cfg.finetuned_model = False
base_encoder.requires_grad_(False)
PATCH_SIZE = base_encoder.input_patch_size
assert 128 % PATCH_SIZE == 0
assert int(CLIP_SECONDS * 1000 / 10) - 2 >= PATCH_SIZE
print(f"Encoder parameters: {sum(p.numel() for p in base_encoder.parameters()):,}; patch: {PATCH_SIZE}")


/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Encoder parameters: 90,311,792; patch: 16


## Recordings and windows

Read `SUBDIRS`, `DIRECTORY`, `CATEGORY`, and `SPLIT` from Intel's manifest. Preserve the original split in the saved metadata: published TRAIN must contain only good welds, and VAL/TEST must each contain good and defective recordings. Cross-validation repartitions **TRAIN + VAL** below; TEST never enters the folds.

Each recording becomes fixed-length windows, with one extra tail window when needed. Audio uses the first channel, is resampled to 16 kHz, and receives the exact normalization in `BEATs.preprocess`. Short recordings fail explicitly. Split recordings before selecting windows so every window of a recording has the same role. Evaluation averages window errors to one score per recording.

In [15]:
def make_recordings():
    if SMOKE_TEST:
        rows = []
        for split, count in [("TRAIN", 10), ("VAL", 20), ("TEST", 10)]:
            for i in range(count):
                rows.append(dict(recording_id=f"{split}/{i}", split=split,
                                 category="Good" if split == "TRAIN" or i < count // 2 else "Synthetic_defect",
                                 group_id=f"{split}/setup_{i // 2}",
                                 path="", duration=CLIP_SECONDS))
        return pd.DataFrame(rows)
    if not MANIFEST.is_file():
        raise FileNotFoundError(f"Set MANIFEST and DATA_ROOT to the extracted Intel dataset: {MANIFEST}")
    manifest = pd.read_csv(MANIFEST)
    required = {"SUBDIRS", "DIRECTORY", "CATEGORY", "SPLIT"}
    if not required.issubset(manifest.columns):
        raise ValueError(f"Manifest needs columns {sorted(required)}")
    rows = []
    for item in manifest.to_dict("records"):
        folder = DATA_ROOT / str(item["SUBDIRS"])
        audio = sorted(p for p in folder.iterdir() if p.suffix.lower() in {".flac", ".wav"})
        if len(audio) != 1:
            raise ValueError(f"Expected one .flac/.wav in {folder}, found {len(audio)}")
        info = sf.info(str(audio[0]))
        if info.duration + 1e-6 < CLIP_SECONDS:
            raise ValueError(f"{audio[0]} is shorter than CLIP_SECONDS={CLIP_SECONDS}")
        rows.append(dict(recording_id=str(item["SUBDIRS"]), group_id=str(item["DIRECTORY"]),
                         split=str(item["SPLIT"]).strip().upper(), category=str(item["CATEGORY"]).strip(),
                         path=str(audio[0].resolve()), duration=info.duration))
    return pd.DataFrame(rows)

recordings = make_recordings()
assert recordings.recording_id.is_unique, "Duplicate recording IDs, possibly across splits."
assert set(recordings.split) == {"TRAIN", "VAL", "TEST"}
assert recordings.loc[recordings.split == "TRAIN", "category"].eq("Good").all()
for split in ["VAL", "TEST"]:
    assert set(recordings.loc[recordings.split == split, "category"].ne("Good")) == {False, True}
if not SMOKE_TEST:
    assert not recordings.path.duplicated().any(), "Audio path shared between recordings."

windows = []
for item in recordings.to_dict("records"):
    last = max(0.0, item["duration"] - CLIP_SECONDS)
    starts = np.arange(0, last + 1e-6, HOP_SECONDS).tolist()
    if last - starts[-1] > 1e-6:
        starts.append(last)
    windows.extend(dict(item, start=start) for start in starts)
windows = pd.DataFrame(windows)
display(pd.crosstab(recordings.split, recordings.category))
print("Windows per split:", windows.groupby("split").size().to_dict())


category,Burnthrough,Crater_Cracks,Excessive_Convexity,Excessive_Penetration,Good,Lack_of_Fusion,Overlap,Porosity,Porosity_w_Excessive_Penetration,Spatter,Undercut,Warping
split,,,,,,,,,,,,
TEST,160,81,80,240,121,160,80,170,240,160,80,160
TRAIN,0,0,0,0,576,0,0,0,0,0,0,0
VAL,160,80,80,240,122,160,80,170,240,160,80,160


Windows per split: {'TEST': 32434, 'TRAIN': 10944, 'VAL': 32428}


In [16]:
def load_window(row):
    n = round(CLIP_SECONDS * SAMPLE_RATE)
    if SMOKE_TEST:
        index = int(row["recording_id"].split("/")[-1])
        split_offset = {"TRAIN": 0, "VAL": 100, "TEST": 200}[row["split"]]
        rng = np.random.default_rng(SEED + index + split_offset)
        t = np.arange(n) / SAMPLE_RATE
        frequency = 300 if row["category"] == "Good" else 1800
        wave = 0.1 * np.sin(2 * np.pi * frequency * t) + 0.01 * rng.standard_normal(n)
        return torch.tensor(wave, dtype=torch.float32)
    with sf.SoundFile(row["path"]) as audio:
        original_sr = audio.samplerate
        start = min(round(row["start"] * original_sr),
                    max(0, len(audio) - round(CLIP_SECONDS * original_sr)))
        audio.seek(start)
        wave = audio.read(round(CLIP_SECONDS * original_sr), dtype="float32", always_2d=True)
    wave = torch.from_numpy(wave[:, 0].copy())
    if original_sr != SAMPLE_RATE:
        wave = torchaudio.functional.resample(wave, original_sr, SAMPLE_RATE)
    if wave.numel() < n:
        raise ValueError(f"Too few samples after resampling: {row['path']}")
    wave = wave[:n]
    if not torch.isfinite(wave).all():
        raise ValueError(f"Non-finite audio: {row['path']}")
    return wave


def normalized_fbank(wave):
    features = ta_kaldi.fbank(wave.unsqueeze(0) * 2 ** 15, num_mel_bins=128,
                             sample_frequency=SAMPLE_RATE, frame_length=25, frame_shift=10)
    return (features - 15.41663) / (2 * 6.55582)


class WeldingWindows(Dataset):
    def __init__(self, frame):
        self.rows = frame.to_dict("records")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        return normalized_fbank(load_window(row)), row["recording_id"]


def make_loader(frame, shuffle=False, seed=SEED):
    selected = windows[windows.recording_id.isin(frame.recording_id)]
    assert set(selected.recording_id) == set(frame.recording_id)
    return DataLoader(WeldingWindows(selected), batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed), num_workers=0,
                      pin_memory=DEVICE.type == "cuda")

sample_wave = load_window(windows.iloc[0].to_dict())
with torch.no_grad():
    torch.testing.assert_close(normalized_fbank(sample_wave),
                               base_encoder.preprocess(sample_wave[None])[0])
print("Preprocessing matches BEATs.preprocess.")


Preprocessing matches BEATs.preprocess.


## Five grouped outer folds with separate tuning data

Create five reproducible, category-stratified folds over the combined TRAIN + VAL development recordings, grouping on `DIRECTORY` (welding setup). A setup and all its recordings/windows stay together. Every development recording is an outer holdout exactly once. The same fold assignments serve both variants.

Within each outer training partition, hold out 20% of its groups for tuning. The remaining **good-only** recordings receive gradient updates; remaining defective recordings are unused. Tuning and outer holdout each require both labels. This separate tuning partition avoids selecting the best epoch or threshold using the outer fold whose performance is reported. The saved assignment table records all roles, including unused defects.

In [17]:
development = recordings[recordings.split.isin(["TRAIN", "VAL"])].reset_index(drop=True)
test_recordings = recordings[recordings.split == "TEST"].reset_index(drop=True)
assert not set(development.recording_id) & set(test_recordings.recording_id)
assert development.group_id.nunique() >= N_FOLDS
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds, assignments = [], []
for fold_number, (fit_idx, holdout_idx) in enumerate(
        outer.split(development, development.category, development.group_id), start=1):
    fit_pool = development.iloc[fit_idx].reset_index(drop=True)
    holdout = development.iloc[holdout_idx].copy()
    inner = GroupShuffleSplit(n_splits=100, test_size=TUNING_FRACTION, random_state=SEED + fold_number)
    for train_idx, tune_idx in inner.split(fit_pool, groups=fit_pool.group_id):
        train_pool, tune = fit_pool.iloc[train_idx].copy(), fit_pool.iloc[tune_idx].copy()
        train = train_pool[train_pool.category == "Good"].copy()
        if len(train) and tune.category.ne("Good").nunique() == 2:
            break
    else:
        raise ValueError(f"Fold {fold_number}: cannot form a good training set and a two-label tuning set.")
    assert holdout.category.ne("Good").nunique() == 2, f"Fold {fold_number} lacks one evaluation label."
    for left, right in [(train_pool, tune), (train_pool, holdout), (tune, holdout)]:
        assert not set(left.recording_id) & set(right.recording_id)
        assert not set(left.group_id) & set(right.group_id)
    folds.append(dict(fold=fold_number, train=train, tune=tune, holdout=holdout))
    for role, frame in [("train_good", train), ("tune", tune), ("holdout", holdout),
                        ("unused_defect", train_pool[train_pool.category != "Good"])]:
        assignments.append(frame.assign(fold=fold_number, role=role))
fold_assignments = pd.concat(assignments, ignore_index=True)
outer_assignments = fold_assignments[fold_assignments.role == "holdout"]
assert outer_assignments.recording_id.is_unique
assert set(outer_assignments.recording_id) == set(development.recording_id)
assert fold_assignments.groupby(["fold", "group_id"]).role.nunique().eq(1).all()
assert fold_assignments.groupby(["fold", "recording_id"]).size().eq(1).all()
assert fold_assignments.groupby("recording_id").fold.nunique().eq(N_FOLDS).all()
display(fold_assignments.groupby(["fold", "role"]).size().unstack(fill_value=0))
print(f"{N_FOLDS} folds; {len(development)} development recordings; {len(test_recordings)} TEST recordings held out.")


role,holdout,train_good,tune,unused_defect
fold,,,,
1,456,462,333,1057
2,461,463,355,1029
3,462,494,346,1006
4,465,454,355,1034
5,464,451,320,1073


5 folds; 2308 development recordings; 1732 TEST recordings held out.


## Identical autoencoders; different trainable weights

The decoder maps each BEATs token through a bottleneck back to its corresponding log-mel patch. `unfold` uses the same patch size, stride, and token order as BEATs' convolution. Incomplete boundary patches are dropped consistently by both operations. Reconstruction loss is mean squared error over all retained patch values.

The frozen encoder stays in evaluation mode while its decoder trains. Fine-tuning enables the encoder's training mode and uses a smaller encoder learning rate. Both runs reset the seed and start from the same encoder and decoder parameters.

In [18]:
class BEATsAutoencoder(nn.Module):
    def __init__(self, encoder, finetune):
        super().__init__()
        self.beats = encoder
        self.finetune = finetune
        self.beats.requires_grad_(finetune)
        self.decoder = nn.Sequential(
            nn.Linear(encoder.cfg.encoder_embed_dim, BOTTLENECK), nn.GELU(),
            nn.Linear(BOTTLENECK, PATCH_SIZE ** 2),
        )

    def train(self, mode=True):
        super().train(mode)
        if not self.finetune:
            self.beats.eval()
        return self

    def encode(self, fbank):
        x = self.beats.patch_embedding(fbank[:, None]).flatten(2).transpose(1, 2)
        x = self.beats.layer_norm(x)
        if self.beats.post_extract_proj is not None:
            x = self.beats.post_extract_proj(x)
        x = self.beats.dropout_input(x)
        return self.beats.encoder(x, padding_mask=None)[0]

    def forward(self, fbank):
        target = F.unfold(fbank[:, None], kernel_size=PATCH_SIZE,
                          stride=PATCH_SIZE).transpose(1, 2)
        if self.finetune:
            tokens = self.encode(fbank)
        else:
            with torch.no_grad():
                tokens = self.encode(fbank)
        reconstruction = self.decoder(tokens)
        assert reconstruction.shape == target.shape
        return (reconstruction - target).square().mean(dim=(1, 2))


def clone_encoder():
    encoder = type(base_encoder)(copy.deepcopy(base_encoder.cfg))
    encoder.load_state_dict(base_encoder.state_dict(), strict=True)
    return encoder.eval()


seed_everything()
template = BEATsAutoencoder(clone_encoder(), finetune=False)
initial_state = copy.deepcopy(template.state_dict())
parameter_signature = [(name, tuple(value.shape)) for name, value in template.named_parameters()]
with torch.no_grad():
    template.eval()
    fbank = normalized_fbank(sample_wave)[None]
    upstream_tokens = base_encoder.extract_features(sample_wave[None])[0]
    torch.testing.assert_close(template.encode(fbank), upstream_tokens)
    assert torch.isfinite(template(fbank)).all()
del template
print("Token extraction matches BEATs.extract_features; reconstruction shapes are valid.")


/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Token extraction matches BEATs.extract_features; reconstruction shapes are valid.


## Train independently within each fold

Both variants restart from the common initial weights in **every fold**. Gradients use only the good recordings in that fold's training subset. Select the best epoch by ROC-AUC on its separate tuning subset, then calibrate its threshold using only good tuning scores. Neither outer holdout scores nor TEST scores select epochs or thresholds.

Architecture, initial weights, epoch budget, and batch ordering match between variants within each fold. Outputs live in `fold_01/` through `fold_05/`, including the best checkpoints, tuning scores, and holdout scores. Every run checks that encoder weights change only for the fine-tuned variant and that decoder weights change for both. Five folds train ten models, so the real experiment costs approximately five times the earlier two-model comparison.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
settings = dict(seed=SEED, smoke_test=SMOKE_TEST, checkpoint=str(CHECKPOINT),
                manifest=str(MANIFEST), sample_rate=SAMPLE_RATE, clip_seconds=CLIP_SECONDS,
                hop_seconds=HOP_SECONDS, batch_size=BATCH_SIZE, epochs=EPOCHS,
                n_folds=N_FOLDS, tuning_fraction=TUNING_FRACTION, group_column="DIRECTORY",
                cv_pool=["TRAIN", "VAL"], encoder_lr=ENCODER_LR, decoder_lr=DECODER_LR,
                weight_decay=WEIGHT_DECAY, bottleneck=BOTTLENECK, target_fpr=TARGET_FPR,
                aggregation="mean", device=str(DEVICE), torch_version=str(torch.__version__))
(OUTPUT_DIR / "settings.json").write_text(json.dumps(settings, indent=2))
recordings.to_csv(OUTPUT_DIR / "recordings.csv", index=False)
windows.to_csv(OUTPUT_DIR / "windows.csv", index=False)
fold_assignments.to_csv(OUTPUT_DIR / "fold_assignments.csv", index=False)

@torch.no_grad()
def score_recordings(model, frame, seed=SEED):
    model.eval()
    scores = []
    for fbanks, ids in make_loader(frame, seed=seed):
        errors = model(fbanks.to(DEVICE)).cpu().numpy()
        scores.extend(zip(ids, errors.tolist()))
    scored = pd.DataFrame(scores, columns=["recording_id", "score"])
    scored = scored.groupby("recording_id", as_index=False).score.mean()
    scored = scored.merge(frame[["recording_id", "category", "split", "group_id"]],
                          on="recording_id", validate="one_to_one")
    scored["label"] = scored.category.ne("Good").astype(int)
    assert set(scored.recording_id) == set(frame.recording_id)
    assert np.isfinite(scored.score).all()
    return scored


def new_model(finetune):
    model = BEATsAutoencoder(clone_encoder(), finetune=finetune)
    model.load_state_dict(initial_state, strict=True)
    assert [(n, tuple(p.shape)) for n, p in model.named_parameters()] == parameter_signature
    return model.to(DEVICE)


def release_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def metric_row(frame, threshold):
    predicted = frame.score > threshold
    return dict(auc=roc_auc_score(frame.label, frame.score),
                pauc_10=roc_auc_score(frame.label, frame.score, max_fpr=0.1),
                ap=average_precision_score(frame.label, frame.score),
                fpr=predicted[frame.label == 0].mean(),
                tpr=predicted[frame.label == 1].mean())


def train_variant(name, finetune, fold):
    fold_number, fold_seed = fold["fold"], SEED + fold["fold"]
    seed_everything(fold_seed)
    model = new_model(finetune)
    groups = [{"params": model.decoder.parameters(), "lr": DECODER_LR}]
    if finetune:
        groups.append({"params": model.beats.parameters(), "lr": ENCODER_LR})
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    train_loader = make_loader(fold["train"], shuffle=True, seed=fold_seed)
    best_auc, history = -np.inf, []
    fold_dir = OUTPUT_DIR / f"fold_{fold_number:02d}"
    fold_dir.mkdir(exist_ok=True)
    checkpoint_path = fold_dir / f"{name}_best.pt"
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total, count = 0.0, 0
        for fbanks, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = model(fbanks.to(DEVICE)).mean()
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss in fold {fold_number}, {name}, epoch {epoch}")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total += loss.item() * len(fbanks)
            count += len(fbanks)
        tune = score_recordings(model, fold["tune"], seed=fold_seed)
        auc = roc_auc_score(tune.label, tune.score)
        history.append(dict(fold=fold_number, variant=name, epoch=epoch,
                            train_mse=total / count, tune_auc=auc))
        print(f"Fold {fold_number}/{N_FOLDS}, {name}: epoch {epoch}/{EPOCHS}, "
              f"train MSE={total/count:.4f}, tuning AUC={auc:.4f}")
        if auc > best_auc:
            best_auc = auc
            threshold = float(np.quantile(tune.loc[tune.label == 0, "score"], 1 - TARGET_FPR))
            torch.save(dict(model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                            cfg=vars(model.beats.cfg), settings=settings, finetune=finetune,
                            fold=fold_number, epoch=epoch, tune_auc=float(auc), threshold=threshold),
                       checkpoint_path)
    changed = any(not torch.equal(p.detach().cpu(), initial_state[n])
                  for n, p in model.named_parameters() if n.startswith("beats."))
    assert changed == finetune, f"Unexpected encoder weight updates for fold {fold_number}, {name}"
    assert any(not torch.equal(p.detach().cpu(), initial_state[n])
               for n, p in model.named_parameters() if n.startswith("decoder."))
    del optimizer, model
    release_memory()
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = new_model(finetune)
    model.load_state_dict(saved["model"], strict=True)
    tune = score_recordings(model, fold["tune"])
    holdout = score_recordings(model, fold["holdout"])
    holdout["prediction"] = (holdout.score > saved["threshold"]).astype(int)
    holdout["fold"], holdout["variant"] = fold_number, name
    tune.to_csv(fold_dir / f"{name}_tuning_scores.csv", index=False)
    holdout.to_csv(fold_dir / f"{name}_holdout_scores.csv", index=False)
    row = dict(fold=fold_number, variant=name, best_epoch=saved["epoch"],
               tune_auc=saved["tune_auc"], threshold=saved["threshold"],
               **metric_row(holdout, saved["threshold"]))
    del model, saved
    release_memory()
    return pd.DataFrame(history), row, holdout

histories, fold_metrics, oof_frames = [], [], []
for fold in folds:
    for name, finetune in [("frozen", False), ("finetuned", True)]:
        run_history, row, holdout = train_variant(name, finetune, fold)
        histories.append(run_history)
        fold_metrics.append(row)
        oof_frames.append(holdout)
history = pd.concat(histories, ignore_index=True)
cv_results = pd.DataFrame(fold_metrics)
oof_predictions = pd.concat(oof_frames, ignore_index=True)
history.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
cv_results.to_csv(OUTPUT_DIR / "cv_fold_metrics.csv", index=False)
oof_predictions.to_csv(OUTPUT_DIR / "out_of_fold_scores.csv", index=False)
for name in ["frozen", "finetuned"]:
    scored = oof_predictions[oof_predictions.variant == name]
    assert scored.recording_id.is_unique
    assert set(scored.recording_id) == set(development.recording_id)
print("All development recordings received exactly one outer-holdout score per variant.")


/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Fold 1/5, frozen: epoch 1/5, train MSE=0.0081, tuning AUC=0.4643
Fold 1/5, frozen: epoch 2/5, train MSE=0.0052, tuning AUC=0.7438
Fold 1/5, frozen: epoch 3/5, train MSE=0.0047, tuning AUC=0.4499
Fold 1/5, frozen: epoch 4/5, train MSE=0.0045, tuning AUC=0.4246
Fold 1/5, frozen: epoch 5/5, train MSE=0.0044, tuning AUC=0.4946


/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Fold 1/5, finetuned: epoch 1/5, train MSE=0.0069, tuning AUC=0.4255
Fold 1/5, finetuned: epoch 2/5, train MSE=0.0037, tuning AUC=0.5636
Fold 1/5, finetuned: epoch 3/5, train MSE=0.0033, tuning AUC=0.4388
Fold 1/5, finetuned: epoch 4/5, train MSE=0.0031, tuning AUC=0.2331
Fold 1/5, finetuned: epoch 5/5, train MSE=0.0029, tuning AUC=0.4706


/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


...

Fold 3/5, finetuned: epoch 1/5, train MSE=0.0068, tuning AUC=0.6520
Fold 3/5, finetuned: epoch 2/5, train MSE=0.0037, tuning AUC=0.6006
Fold 3/5, finetuned: epoch 3/5, train MSE=0.0034, tuning AUC=0.4296
Fold 3/5, finetuned: epoch 4/5, train MSE=0.0031, tuning AUC=0.4223
Fold 3/5, finetuned: epoch 5/5, train MSE=0.0029, tuning AUC=0.6010


/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Fold 4/5, frozen: epoch 1/5, train MSE=0.0084, tuning AUC=0.6797
Fold 4/5, frozen: epoch 2/5, train MSE=0.0052, tuning AUC=0.7526
Fold 4/5, frozen: epoch 3/5, train MSE=0.0049, tuning AUC=0.6901
Fold 4/5, frozen: epoch 4/5, train MSE=0.0046, tuning AUC=0.6432
Fold 4/5, frozen: epoch 5/5, train MSE=0.0045, tuning AUC=0.6992


/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/jaden/Documents/ai-360/AI360-Project/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Fold 4/5, finetuned: epoch 1/5, train MSE=0.0072, tuning AUC=0.5759


## Five-fold results and final TEST evaluation

Report each outer holdout and the **mean ± sample standard deviation** across the five folds. Pair each fine-tuned result with its frozen baseline on the same holdout. Scores from separately trained folds can have different scales, so aggregate metrics per fold rather than calculate one AUC from pooled raw reconstruction errors.

Only after every fold finishes training, evaluate its selected models on the original TEST recordings with the stored tuning thresholds. These ten evaluations do not change any checkpoint, threshold, or settings. The TEST summary describes the five fitted models on the same fixed test set; its standard deviation is not an independent-sample confidence interval. Per-defect CV AUC compares each available defect category with the good recordings in the same outer holdout.

In [ ]:
METRIC_COLUMNS = ["auc", "pauc_10", "ap", "fpr", "tpr"]
cv_summary = cv_results.groupby("variant")[METRIC_COLUMNS].agg(["mean", "std"])
paired = cv_results.pivot(index="fold", columns="variant", values=METRIC_COLUMNS)
paired_differences = pd.DataFrame({metric: paired[metric]["finetuned"] - paired[metric]["frozen"]
                                   for metric in METRIC_COLUMNS})
paired_summary = paired_differences.agg(["mean", "std"])
cv_summary.to_csv(OUTPUT_DIR / "cv_summary.csv")
paired_differences.to_csv(OUTPUT_DIR / "cv_paired_differences.csv")
paired_summary.to_csv(OUTPUT_DIR / "cv_paired_summary.csv")
display(cv_results.round(4))
display(cv_summary.round(4))
print("Paired differences: finetuned - frozen (positive favors fine-tuning except for FPR)")
display(paired_summary.round(4))

defect_metrics = []
for (fold_number, name), scored in oof_predictions.groupby(["fold", "variant"]):
    for category in sorted(set(scored.category) - {"Good"}):
        subset = scored[scored.category.isin(["Good", category])]
        defect_metrics.append(dict(fold=fold_number, variant=name, category=category,
                                   auc=roc_auc_score(subset.label, subset.score)))
per_defect = pd.DataFrame(defect_metrics)
per_defect.to_csv(OUTPUT_DIR / "cv_per_defect_fold_auc.csv", index=False)
per_defect_summary = per_defect.groupby(["category", "variant"]).auc.agg(["mean", "std", "count"])
per_defect_summary.to_csv(OUTPUT_DIR / "cv_per_defect_summary.csv")
display(per_defect_summary.round(4))

test_metrics, test_frames = [], []
for fold in folds:
    for name, finetune in [("frozen", False), ("finetuned", True)]:
        fold_dir = OUTPUT_DIR / f"fold_{fold['fold']:02d}"
        saved = torch.load(fold_dir / f"{name}_best.pt", map_location="cpu", weights_only=True)
        model = new_model(finetune)
        model.load_state_dict(saved["model"], strict=True)
        test = score_recordings(model, test_recordings)
        test["fold"], test["variant"] = fold["fold"], name
        test["prediction"] = (test.score > saved["threshold"]).astype(int)
        test_frames.append(test)
        test_metrics.append(dict(fold=fold["fold"], variant=name,
                                 **metric_row(test, saved["threshold"])))
        test.to_csv(fold_dir / f"{name}_test_scores.csv", index=False)
        del model, saved
        release_memory()
test_predictions = pd.concat(test_frames, ignore_index=True)
test_results = pd.DataFrame(test_metrics)
test_summary = test_results.groupby("variant")[METRIC_COLUMNS].agg(["mean", "std"])
test_results.to_csv(OUTPUT_DIR / "test_fold_metrics.csv", index=False)
test_summary.to_csv(OUTPUT_DIR / "test_summary.csv")
print("Final published TEST split:")
display(test_summary.round(4))
if SMOKE_TEST:
    print("SYNTHETIC SMOKE TEST ONLY: these scores do not measure pretrained BEATs performance.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in ["frozen", "finetuned"]:
    run = history[history.variant == name].groupby("epoch").train_mse.agg(["mean", "std"])
    axes[0].plot(run.index, run["mean"], marker="o", label=name)
    axes[0].fill_between(run.index, run["mean"] - run["std"],
                         run["mean"] + run["std"], alpha=0.15)
    scores = cv_results[cv_results.variant == name].sort_values("fold")
    axes[1].plot(scores.fold, scores.auc, marker="o", label=name)
    grid, curves = np.linspace(0, 1, 101), []
    for _, test in test_predictions[test_predictions.variant == name].groupby("fold"):
        fpr, tpr, _ = roc_curve(test.label, test.score)
        curve = np.interp(grid, fpr, tpr)
        curve[0], curve[-1] = 0.0, 1.0
        curves.append(curve)
    mean_tpr, std_tpr = np.mean(curves, axis=0), np.std(curves, axis=0, ddof=1)
    auc = test_summary.loc[name, ("auc", "mean")]
    axes[2].plot(grid, mean_tpr, label=f"{name} (mean AUC={auc:.3f})")
    axes[2].fill_between(grid, np.maximum(0, mean_tpr - std_tpr),
                         np.minimum(1, mean_tpr + std_tpr), alpha=0.15)
axes[0].set(xlabel="Epoch", ylabel="Training MSE (fold mean ± SD)")
axes[1].set(xlabel="Outer fold", ylabel="Holdout recording ROC-AUC", xticks=range(1, N_FOLDS + 1))
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[2].set(xlabel="False positive rate", ylabel="True positive rate", title="Final TEST ROC (fold mean ± SD)")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "comparison.png", dpi=160)
plt.show()
print(f"Saved fold assignments, ten checkpoints, histories, score tables, summaries, and plot to {OUTPUT_DIR}")


## Reading the result

Use outer-holdout mean ROC-AUC, partial AUC, and paired fold differences for the development comparison. Lower training loss alone does not establish better anomaly detection: a model can also reconstruct defects well. Fold standard deviations describe variation across these splits, not a confidence interval. Per-defect summaries include the number of contributing folds because a category may be absent from an outer holdout.

Five-fold CV here repartitions the original TRAIN + VAL recordings by `DIRECTORY`; it does not reproduce the original paper's single TRAIN/VAL protocol. Good recordings in the original VAL split may therefore enter a fold's training subset. Defective recordings never receive gradient updates. The published TEST split remains untouched by CV construction and tuning, although its welding directories can overlap the development pool as in the original dataset split. Generalization to entirely new test setups requires a separately defined group holdout.

The experiment uses offline per-recording mean window errors, not streaming latency. Smoke mode executes all five folds and both variants with tiny synthetic inputs, checking split isolation, common initialization, preprocessing, gradient updates, checkpoint reloads, and reporting. Real results require running the notebook with `SMOKE_TEST = False`.